# Feature Engineering

In [41]:
import pandas as pd

In [42]:
cleaned_data = pd.read_parquet("../data/cleaned_data.parquet")

In [43]:
cleaned_data["FL_DATE"] = pd.to_datetime(cleaned_data["FL_DATE"], format="%m/%d/%Y %I:%M:%S %p")

In [44]:
cleaned_data["YEAR"] = cleaned_data["FL_DATE"].dt.year
cleaned_data["MONTH"] = cleaned_data["FL_DATE"].dt.month
cleaned_data["DAY_OF_MONTH"] = cleaned_data["FL_DATE"].dt.day
cleaned_data["DAY_OF_WEEK"] = cleaned_data["FL_DATE"].dt.dayofweek

In [45]:
cleaned_data[["DAY_OF_WEEK", "MONTH", "DAY_OF_MONTH", "YEAR"]]

,DAY_OF_WEEK,MONTH,DAY_OF_MONTH,YEAR
0,0,9,1,2025
1,0,9,1,2025
2,0,9,1,2025
3,0,9,1,2025
4,0,9,1,2025
...,...,...,...,...
9064180,0,6,30,2025
9064181,0,6,30,2025
9064182,0,6,30,2025
9064183,0,6,30,2025


In [46]:
cleaned_data["DEPARTURE_HOUR"] = cleaned_data["CRS_DEP_TIME"] // 100
cleaned_data["DEPARTURE_MINUTE"] = cleaned_data["CRS_DEP_TIME"] % 100
cleaned_data = cleaned_data.drop(columns=["CRS_DEP_TIME"])

In [47]:
cleaned_data["ROUTE"] = cleaned_data["ORIGIN"] + "-" + cleaned_data["DEST"]

In [48]:
def isHolidayPeriod(date):
    # MLK Day
    if date.month == 1 and 15 <= date.day <= 20:
        return 1

    # Presidents Day
    if date.month == 2 and 13 <= date.day <= 17:
        return 1
    
    # Spring Break
    if (date.month == 3 and date.day >= 15) or (date.month == 4 and date.day <= 15):
        return 1
    
    # Memorial Day 
    if date.month == 5 and 23 <= date.day <= 27:
        return 1
    
    # Fourth of July
    if date.month == 7 and 1 <= date.day <= 7:
        return 1
    
    # Labor Day
    if (date.month == 8 and date.day >= 29) or (date.month == 9 and date.day <= 2):
        return 1
    
    # Thanksgiving (Nov 20 – Dec 1)
    if (date.month == 11 and date.day >= 20) or (date.month == 12 and date.day <= 1):
        return 1

    # Christmas & New Year (Dec 20 – Jan 5)
    if (date.month == 12 and date.day >= 20) or (date.month == 1 and date.day <= 5):
        return 1

    return 0

In [49]:
cleaned_data['IS_HOLIDAY_PERIOD'] = cleaned_data['FL_DATE'].apply(isHolidayPeriod)

In [50]:
cleaned_data['DELAYED'] = (cleaned_data['DEP_DELAY'] > 15).astype(int)

In [51]:
cleaned_data['ORIGIN_HOURLY_FLIGHTS'] = cleaned_data.groupby(['ORIGIN', 'FL_DATE', 'DEPARTURE_HOUR'])['ORIGIN'].transform('count') # ORIGIN_HOURLY_FLIGHTS - How many flights departed from that origin airport in that hour

cleaned_data['ROUTE_HOURLY_FLIGHTS'] = cleaned_data.groupby(['ROUTE', 'FL_DATE', 'DEPARTURE_HOUR'])['ROUTE'].transform('count') # ROUTE_HOURLY_FLIGHTS - How many flights operated on that route in that hour

cleaned_data['DEST_HOURLY_FLIGHTS'] = cleaned_data.groupby(['DEST', 'FL_DATE', 'DEPARTURE_HOUR'])['DEST'].transform('count') # ROUTE_HOURLY_FLIGHTS - How many flights are at the destination at that hour

In [52]:
cleaned_data['DATE'] = cleaned_data['FL_DATE'].dt.date

In [53]:
from sklearn.preprocessing import LabelEncoder
import joblib

In [54]:
cleaned_data[['OP_UNIQUE_CARRIER', 'ORIGIN_STATE_ABR', 'DEST_STATE_ABR']].head()

,OP_UNIQUE_CARRIER,ORIGIN_STATE_ABR,DEST_STATE_ABR
0,AA,NY,CA
1,AA,CA,NY
2,AA,WI,NC
3,AA,NC,MO
4,AA,MO,NC


In [55]:
le_carrier = LabelEncoder()
le_origin_state = LabelEncoder()
le_dest_state = LabelEncoder()

cleaned_data['OP_UNIQUE_CARRIER'] = le_carrier.fit_transform(cleaned_data['OP_UNIQUE_CARRIER'])
cleaned_data['ORIGIN_STATE_ABR'] = le_origin_state.fit_transform(cleaned_data['ORIGIN_STATE_ABR'])
cleaned_data['DEST_STATE_ABR'] = le_dest_state.fit_transform(cleaned_data['DEST_STATE_ABR'])

In [56]:
joblib.dump(le_carrier, '../encodings/le_carrier.pkl')
joblib.dump(le_origin_state, '../encodings/le_origin_state.pkl')
joblib.dump(le_dest_state, '../encodings/le_dest_state.pkl')

['../encodings/le_dest_state.pkl']

In [57]:
y = cleaned_data["DELAYED"]

In [58]:
columns = cleaned_data.corr(numeric_only=True, method='spearman')['DELAYED'].sort_values(ascending=False)
columns

DELAYED                  1.000000
DEP_DEL15                0.980188
DEP_DELAY_NEW            0.807794
ARR_DEL15                0.745245
DEP_DELAY                0.703921
ARR_DELAY                0.648015
LATE_AIRCRAFT_DELAY      0.636449
CARRIER_DELAY            0.583556
NAS_DELAY                0.346173
WEATHER_DELAY            0.203604
DEPARTURE_HOUR           0.197246
DEPARTURE_MINUTE         0.045239
SECURITY_DELAY           0.044134
ORIGIN_HOURLY_FLIGHTS    0.033193
DAY_OF_WEEK              0.028636
MONTH                    0.023176
IS_HOLIDAY_PERIOD        0.019701
DISTANCE                 0.015500
CRS_ELAPSED_TIME         0.014809
DISTANCE_GROUP           0.014104
ORIGIN_STATE_ABR         0.007231
DEST_STATE_ABR           0.005324
DAY_OF_MONTH             0.001495
OP_CARRIER_FL_NUM        0.000951
ROUTE_HOURLY_FLIGHTS    -0.000490
YEAR                    -0.002175
OP_UNIQUE_CARRIER       -0.008398
DEST_HOURLY_FLIGHTS     -0.036814
CANCELLED                     NaN
DIVERTED      

In [59]:
cleaned_data = cleaned_data.drop(columns=['DEP_DELAY_NEW', 'DEP_DELAY', 'LATE_AIRCRAFT_DELAY', 'CARRIER_DELAY', 'NAS_DELAY', 'WEATHER_DELAY', 'SECURITY_DELAY'])

In [60]:
updated_correlation_matrix = cleaned_data.corr(numeric_only=True, method='spearman')
updated_numerical_columns = updated_correlation_matrix['DELAYED'].sort_values(ascending=False)
updated_numerical_columns

DELAYED                  1.000000
DEP_DEL15                0.980188
ARR_DEL15                0.745245
ARR_DELAY                0.648015
DEPARTURE_HOUR           0.197246
DEPARTURE_MINUTE         0.045239
ORIGIN_HOURLY_FLIGHTS    0.033193
DAY_OF_WEEK              0.028636
MONTH                    0.023176
IS_HOLIDAY_PERIOD        0.019701
DISTANCE                 0.015500
CRS_ELAPSED_TIME         0.014809
DISTANCE_GROUP           0.014104
ORIGIN_STATE_ABR         0.007231
DEST_STATE_ABR           0.005324
DAY_OF_MONTH             0.001495
OP_CARRIER_FL_NUM        0.000951
ROUTE_HOURLY_FLIGHTS    -0.000490
YEAR                    -0.002175
OP_UNIQUE_CARRIER       -0.008398
DEST_HOURLY_FLIGHTS     -0.036814
CANCELLED                     NaN
DIVERTED                      NaN
Name: DELAYED, dtype: float64

In [61]:
updated_numerical_columns = updated_numerical_columns.drop(['DELAYED', 'DEPARTURE_MINUTE', 'DISTANCE_GROUP', 'OP_CARRIER_FL_NUM', 'DAY_OF_MONTH', 'YEAR'])
updated_numerical_columns

DEP_DEL15                0.980188
ARR_DEL15                0.745245
ARR_DELAY                0.648015
DEPARTURE_HOUR           0.197246
ORIGIN_HOURLY_FLIGHTS    0.033193
DAY_OF_WEEK              0.028636
MONTH                    0.023176
IS_HOLIDAY_PERIOD        0.019701
DISTANCE                 0.015500
CRS_ELAPSED_TIME         0.014809
ORIGIN_STATE_ABR         0.007231
DEST_STATE_ABR           0.005324
ROUTE_HOURLY_FLIGHTS    -0.000490
OP_UNIQUE_CARRIER       -0.008398
DEST_HOURLY_FLIGHTS     -0.036814
CANCELLED                     NaN
DIVERTED                      NaN
Name: DELAYED, dtype: float64

In [62]:
features = updated_numerical_columns.index.tolist()

In [63]:
features.append('ORIGIN')
features.append('DEST')
features.append('ROUTE')

In [64]:
features

['DEP_DEL15',
 'ARR_DEL15',
 'ARR_DELAY',
 'DEPARTURE_HOUR',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'CANCELLED',
 'DIVERTED',
 'ORIGIN',
 'DEST',
 'ROUTE']

In [65]:
cols = ['ARR_DEL15', 'ARR_DELAY', 'DEP_DEL15', 'CANCELLED', 'DIVERTED']
for col in cols:
    features.remove(col)

In [66]:
features

['DEPARTURE_HOUR',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'ORIGIN',
 'DEST',
 'ROUTE']

In [67]:
X = cleaned_data[features]

In [68]:
from sklearn.model_selection import train_test_split

In [69]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=1234)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=1234)

In [70]:
carrier_delay_map = pd.concat([X_train, y_train], axis=1).groupby('OP_UNIQUE_CARRIER')['DELAYED'].mean()
origin_delay_map = pd.concat([X_train, y_train], axis=1).groupby('ORIGIN')['DELAYED'].mean()
route_delay_map = pd.concat([X_train, y_train], axis=1).groupby('ROUTE')['DELAYED'].mean()
dest_delay_map = pd.concat([X_train, y_train], axis=1).groupby('DEST')['DELAYED'].mean()

# CARRIER_DELAY_RATE - percentage of flights that were delayed for a specific airline. 
X_train['CARRIER_DELAY_RATE'] = X_train['OP_UNIQUE_CARRIER'].map(carrier_delay_map)
X_val['CARRIER_DELAY_RATE'] = X_val['OP_UNIQUE_CARRIER'].map(carrier_delay_map)
X_test['CARRIER_DELAY_RATE'] = X_test['OP_UNIQUE_CARRIER'].map(carrier_delay_map)

# ORIGIN_DELAY_RATE - percentage of flights that were delayed departing from a specific airport. 
X_train['ORIGIN_DELAY_RATE'] = X_train['ORIGIN'].map(origin_delay_map)
X_val['ORIGIN_DELAY_RATE'] = X_val['ORIGIN'].map(origin_delay_map)
X_test['ORIGIN_DELAY_RATE'] = X_test['ORIGIN'].map(origin_delay_map)

# ROUTE_DELAY_RATE - percentage of flights that were delayed on a specific route. 
X_train['ROUTE_DELAY_RATE'] = X_train['ROUTE'].map(route_delay_map)
X_val['ROUTE_DELAY_RATE'] = X_val['ROUTE'].map(route_delay_map)
X_test['ROUTE_DELAY_RATE'] = X_test['ROUTE'].map(route_delay_map)

X_train['DEST_DELAY_RATE'] = X_train['DEST'].map(dest_delay_map)
X_val['DEST_DELAY_RATE'] = X_val['DEST'].map(dest_delay_map)
X_test['DEST_DELAY_RATE'] = X_test['DEST'].map(dest_delay_map)

In [71]:
route_mean = route_delay_map.mean()
X_val['ROUTE_DELAY_RATE'] = X_val['ROUTE_DELAY_RATE'].fillna(route_mean)
X_test['ROUTE_DELAY_RATE'] = X_test['ROUTE_DELAY_RATE'].fillna(route_mean)

In [72]:
dest_mean = dest_delay_map.mean()
X_val['DEST_DELAY_RATE'] = X_val['DEST_DELAY_RATE'].fillna(dest_mean)
X_test['DEST_DELAY_RATE'] = X_test['DEST_DELAY_RATE'].fillna(dest_mean)

In [73]:
from sklearn.preprocessing import TargetEncoder

In [74]:
te_origin = TargetEncoder()
te_dest = TargetEncoder()
te_route = TargetEncoder()

In [75]:
# Fit transform the training data
X_train['ORIGIN_ENCODED'] = te_origin.fit_transform(X_train[['ORIGIN']], y_train)
X_train['DEST_ENCODED'] = te_dest.fit_transform(X_train[['DEST']], y_train)
X_train['ROUTE_ENCODED'] = te_route.fit_transform(X_train[['ROUTE']], y_train)

X_val['ORIGIN_ENCODED'] = te_origin.transform(X_val[['ORIGIN']])
X_val['DEST_ENCODED'] = te_dest.transform(X_val[['DEST']])
X_val['ROUTE_ENCODED'] = te_route.transform(X_val[['ROUTE']])

X_test['ORIGIN_ENCODED'] = te_origin.transform(X_test[['ORIGIN']])
X_test['DEST_ENCODED'] = te_dest.transform(X_test[['DEST']])
X_test['ROUTE_ENCODED'] = te_route.transform(X_test[['ROUTE']])

X_train = X_train.drop(columns=['ORIGIN', 'DEST', 'ROUTE'])
X_val = X_val.drop(columns=['ORIGIN', 'DEST', 'ROUTE'])
X_test = X_test.drop(columns=['ORIGIN', 'DEST', 'ROUTE'])

In [76]:
print('OP_UNIQUE_CARRIER' in X_train.columns)

True


In [77]:
# Saving the origin, dest, and route target encoders
joblib.dump(te_origin, '../encodings/origin_te.pkl')
joblib.dump(te_dest, '../encodings/dest_te.pkl')
joblib.dump(te_route, '../encodings/route_te.pkl')

['../encodings/route_te.pkl']

In [78]:
# Saving the training, validation, and testing data
X_train.to_parquet('../data/X_train.parquet', index=False)
X_val.to_parquet('../data/X_val.parquet', index=False)
X_test.to_parquet('../data/X_test.parquet', index=False)
y_train.to_frame().to_parquet('../data/y_train.parquet', index=False)
y_val.to_frame().to_parquet('../data/y_val.parquet', index=False)
y_test.to_frame().to_parquet('../data/y_test.parquet', index=False)

In [80]:
X_train.columns.tolist()

['DEPARTURE_HOUR',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'CARRIER_DELAY_RATE',
 'ORIGIN_DELAY_RATE',
 'ROUTE_DELAY_RATE',
 'DEST_DELAY_RATE',
 'ORIGIN_ENCODED',
 'DEST_ENCODED',
 'ROUTE_ENCODED']

In [81]:
X_val.columns.tolist()

['DEPARTURE_HOUR',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'CARRIER_DELAY_RATE',
 'ORIGIN_DELAY_RATE',
 'ROUTE_DELAY_RATE',
 'DEST_DELAY_RATE',
 'ORIGIN_ENCODED',
 'DEST_ENCODED',
 'ROUTE_ENCODED']

In [82]:
X_test.columns.tolist()

['DEPARTURE_HOUR',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'CARRIER_DELAY_RATE',
 'ORIGIN_DELAY_RATE',
 'ROUTE_DELAY_RATE',
 'DEST_DELAY_RATE',
 'ORIGIN_ENCODED',
 'DEST_ENCODED',
 'ROUTE_ENCODED']

In [84]:
X_train.dtypes

DEPARTURE_HOUR             int64
ORIGIN_HOURLY_FLIGHTS      int64
DAY_OF_WEEK                int32
MONTH                      int32
IS_HOLIDAY_PERIOD          int64
DISTANCE                 float64
CRS_ELAPSED_TIME         float64
ORIGIN_STATE_ABR           int64
DEST_STATE_ABR             int64
ROUTE_HOURLY_FLIGHTS       int64
OP_UNIQUE_CARRIER          int64
DEST_HOURLY_FLIGHTS        int64
CARRIER_DELAY_RATE       float64
ORIGIN_DELAY_RATE        float64
ROUTE_DELAY_RATE         float64
DEST_DELAY_RATE          float64
ORIGIN_ENCODED           float64
DEST_ENCODED             float64
ROUTE_ENCODED            float64
dtype: object

In [85]:
X_val.dtypes

DEPARTURE_HOUR             int64
ORIGIN_HOURLY_FLIGHTS      int64
DAY_OF_WEEK                int32
MONTH                      int32
IS_HOLIDAY_PERIOD          int64
DISTANCE                 float64
CRS_ELAPSED_TIME         float64
ORIGIN_STATE_ABR           int64
DEST_STATE_ABR             int64
ROUTE_HOURLY_FLIGHTS       int64
OP_UNIQUE_CARRIER          int64
DEST_HOURLY_FLIGHTS        int64
CARRIER_DELAY_RATE       float64
ORIGIN_DELAY_RATE        float64
ROUTE_DELAY_RATE         float64
DEST_DELAY_RATE          float64
ORIGIN_ENCODED           float64
DEST_ENCODED             float64
ROUTE_ENCODED            float64
dtype: object

In [88]:
X_test.dtypes

DEPARTURE_HOUR             int64
ORIGIN_HOURLY_FLIGHTS      int64
DAY_OF_WEEK                int32
MONTH                      int32
IS_HOLIDAY_PERIOD          int64
DISTANCE                 float64
CRS_ELAPSED_TIME         float64
ORIGIN_STATE_ABR           int64
DEST_STATE_ABR             int64
ROUTE_HOURLY_FLIGHTS       int64
OP_UNIQUE_CARRIER          int64
DEST_HOURLY_FLIGHTS        int64
CARRIER_DELAY_RATE       float64
ORIGIN_DELAY_RATE        float64
ROUTE_DELAY_RATE         float64
DEST_DELAY_RATE          float64
ORIGIN_ENCODED           float64
DEST_ENCODED             float64
ROUTE_ENCODED            float64
dtype: object